In [1]:
# ============================================================
# CafeCorp — Akton Partners recruiting case
# Step 1: load, profile, clean
# ============================================================
import pandas as pd

RAW = 'CafeCorp - Sample Database.xlsx'
SHEETS = ['Locations','Products','Channels','Sales','Tickets','Expenses','Inventory','Targets']
d = {sh: pd.read_excel(RAW, sheet_name=sh) for sh in SHEETS}

In [2]:
# ---- 1. grain: does each assumed key identify one row? ----
KEYS = {
    'Locations': ['location_id'],
    'Products':  ['product_id'],
    'Channels':  ['channel'],
    'Sales':     ['date','location_id','channel','product_id'],
    'Tickets':   ['date','location_id','channel'],
    'Expenses':  ['month','location_id'],
    'Inventory': ['month','location_id','item_id'],
    'Targets':   ['month','location_id','channel','category'],
}
for sh, k in KEYS.items():
    print(f"{sh:10s} rows={len(d[sh]):7d}  dupes={d[sh].duplicated(k).sum()}")

Locations  rows=     16  dupes=0
Products   rows=     28  dupes=0
Channels   rows=      3  dupes=0
Sales      rows= 198685  dupes=0
Tickets    rows=  26288  dupes=0
Expenses   rows=    292  dupes=0
Inventory  rows=   4380  dupes=0
Targets    rows=   2628  dupes=0


In [3]:
# ---- 2. referential integrity ----
for col, dim in [('product_id','Products'), ('location_id','Locations'), ('channel','Channels')]:
    orphans = (~d['Sales'][col].isin(d[dim][col])).sum()
    print(f"sales {col} not in {dim}: {orphans}")

sales product_id not in Products: 0
sales location_id not in Locations: 0
sales channel not in Channels: 0


In [4]:
# ---- 3. nulls ----
for sh in SHEETS:
    n = d[sh].isna().sum()
    if n.sum():
        print(sh, dict(n[n > 0]))

Products {'standard_unit_cost': 2}
Expenses {'labor_cost': 1, 'rent': 4}


In [10]:
# ---- 4a. clean: impute product costs (flag first, then fill) ----
P = d['Products'].copy()
P['cost_ratio'] = P.standard_unit_cost / P.unit_price   # from complete rows only
P['cost_imputed'] = P.standard_unit_cost.isna()

cat_ratio = P.groupby('category').cost_ratio.median()
P['standard_unit_cost'] = P.standard_unit_cost.fillna(
    (P.unit_price * P.category.map(cat_ratio)).round(2)
)

d['Products'] = P

print(P.loc[P.cost_imputed, ['product_id','product_name','category',
                             'unit_price','standard_unit_cost']].to_string())

   product_id  product_name   category  unit_price  standard_unit_cost
16       P017  Matcha Latte  Beverages          82               27.47
27       P028  Granola Bowl       Food          98               43.61


In [ ]:
#Check on Expense missing value
S = d['Sales'].copy()
S['month'] = S.date.dt.strftime('%Y-%m')

net_by_month_loc = S.groupby(['month','location_id']).net_sales.sum().reset_index()
lab = d['Expenses'].query("location_id == 'L02'")[['month','labor_cost']]

l02 = net_by_month_loc.query("location_id == 'L02'")[['month','net_sales']].merge(lab, on='month', how='left')
l02['ratio'] = l02.labor_cost / l02.net_sales
print(l02.to_string())

      month   net_sales  labor_cost     ratio
0   2024-01   999568.61   265306.15  0.265421
1   2024-02   917945.45   241231.41  0.262795
2   2024-03  1044091.69   280823.05  0.268964
3   2024-04  1062379.35   282167.51  0.265600
4   2024-05  1099978.50   290101.54  0.263734
5   2024-06  1065420.15   294117.51  0.276058
6   2024-07  1089103.04   300607.72  0.276014
7   2024-08  1053107.62   283815.37  0.269503
8   2024-09   999796.64   274232.29  0.274288
9   2024-10  1099889.46   292080.51  0.265554
10  2024-11  1086487.65   301957.45  0.277921
11  2024-12  1252873.06   335984.00  0.268171
12  2025-01  1051885.43   291645.25  0.277260
13  2025-02   930194.82   256735.68  0.276002
14  2025-03  1099760.97   292744.20  0.266189
15  2025-04  1088167.14   294179.45  0.270344
16  2025-05  1180648.88   310005.26  0.262572
17  2025-06  1171666.24         NaN       NaN
18  2025-07  1140701.58   315157.82  0.276284
19  2025-08  1163312.26   315450.97  0.271166
20  2025-09  1105429.59   298120.7

In [24]:
# ---- 4. clean: impute expenses (flag first, then fill) ----
E = d['Expenses'].copy()
COST_COLS = ['labor_cost','rent','utilities','marketing','other_opex']
for c in COST_COLS:
    E[c + '_imputed'] = E[c].isna()

# rent: fixed contractual cost -> location median
E['rent'] = E.groupby('location_id')['rent'].transform(lambda x: x.fillna(x.median()))

# labour: scales with trading volume -> location's median labour-to-revenue ratio
sm = (d['Sales'].assign(month=lambda x: x.date.dt.strftime('%Y-%m'))
        .groupby(['month','location_id'], as_index=False).net_sales.sum())
E = E.merge(sm, on=['month','location_id'], how='left')

ratio = (E.loc[~E.labor_cost_imputed]
           .assign(r=lambda x: x.labor_cost / x.net_sales)
           .groupby('location_id').r.median())

m = E.labor_cost.isna()
E.loc[m, 'labor_cost'] = (E.loc[m, 'net_sales'] * E.loc[m, 'location_id'].map(ratio)).round(2)

d['Expenses'] = E

E['opex'] = E[COST_COLS].sum(axis=1)
print(E[E[[c + '_imputed' for c in COST_COLS]].any(axis=1)])

       month location_id  labor_cost        rent  utilities  marketing  \
151  2025-03         L05   282943.70  150172.885   46555.98   33691.43   
188  2025-06         L02   315983.91  149594.820   54681.36   31918.70   
205  2025-07         L05   317568.60  150172.885   52443.69   32760.53   
249  2025-10         L05   306505.44  150172.885   52699.41   32947.68   
264  2025-11         L05   316100.56  150172.885   53778.74   37917.88   

     other_opex  labor_cost_imputed  rent_imputed  utilities_imputed  \
151    38274.56               False          True              False   
188    42831.25                True         False              False   
205    40002.95               False          True              False   
249    41153.27               False          True              False   
264    41729.33               False          True              False   

     marketing_imputed  other_opex_imputed   net_sales        opex  
151              False               False  1077136.7

In [30]:
#Check Final insertions
E = d['Expenses']
E[(E.location_id == 'L05') & (E.month.isin(['2025-03','2025-07','2025-10', '2025-11']))]
#E[(E.location_id == 'L02') & (E.month.isin(['2025-06']))]


,month,location_id,labor_cost,rent,utilities,marketing,other_opex,labor_cost_imputed,rent_imputed,utilities_imputed,marketing_imputed,other_opex_imputed,net_sales,opex
151,2025-03,L05,282943.70,150172.885,46555.98,33691.43,38274.56,False,True,False,False,False,1077136.79,551638.555
205,2025-07,L05,317568.60,150172.885,52443.69,32760.53,40002.95,False,True,False,False,False,1145990.50,592948.655
249,2025-10,L05,306505.44,150172.885,52699.41,32947.68,41153.27,False,True,False,False,False,1133030.52,583478.685
264,2025-11,L05,316100.56,150172.885,53778.74,37917.88,41729.33,False,True,False,False,False,1153013.30,599699.395


In [35]:
def get_sales(S, start, end):
    return S[(S.date >= start) & (S.date <= end)]

def enrich_sales(S, P, CH):
    x = S.merge(P[['product_id','standard_unit_cost','category']], on='product_id', how='left')
    x = x.merge(CH[['channel','commission_rate']], on='channel', how='left')
    x['cogs'] = x.quantity * x.standard_unit_cost
    x['commission'] = x.net_sales * x.commission_rate
    return x

enriched = enrich_sales(d['Sales'], P, d['Channels'])
print(enriched[['date','location_id','channel','product_id','net_sales','cogs','commission']].head())

        date location_id   channel product_id  net_sales    cogs  commission
0 2024-01-01         L01  Delivery       P003    1365.00  354.90     382.200
1 2024-01-01         L01  Delivery       P004    1463.15  490.00     409.682
2 2024-01-01         L01  Delivery       P010    1296.00  375.84     362.880
3 2024-01-01         L01  Delivery       P018     245.00   98.00      68.600
4 2024-01-01         L01  Delivery       P021     468.00  205.92     131.040


In [37]:
def income_statement(enriched, E, start, end, locations=None):
    s = enriched[(enriched.date >= start) & (enriched.date <= end)]
    if locations is not None:
        s = s[s.location_id.isin(locations)]

    revenue    = s.net_sales.sum()
    cogs       = s.cogs.sum()
    gross_profit = revenue - cogs
    commission = s.commission.sum()
    contribution = gross_profit - commission

    e = E[(E.month >= start[:7]) & (E.month <= end[:7])]
    if locations is not None:
        e = e[e.location_id.isin(locations)]
    opex = e.opex.sum()

    operating_profit = contribution - opex

    return {
        'revenue': revenue, 'cogs': cogs, 'gross_profit': gross_profit,
        'gross_margin_pct': 100 * gross_profit / revenue,
        'commission': commission, 'contribution': contribution,
        'contribution_pct': 100 * contribution / revenue,
        'opex': opex, 'operating_profit': operating_profit,
        'operating_margin_pct': 100 * operating_profit / revenue,
    }

pl_2025 = income_statement(enriched, E, '2025-01-01', '2025-12-31')
for k, v in pl_2025.items():
    print(f"{k:20s} {v:,.2f}")

revenue              183,001,462.98
cogs                 61,434,534.97
gross_profit         121,566,928.01
gross_margin_pct     66.43
commission           15,057,291.17
contribution         106,509,636.84
contribution_pct     58.20
opex                 90,171,273.64
operating_profit     16,338,363.20
operating_margin_pct 8.93


In [38]:
same_store = [l for l in d['Locations'].location_id if d['Locations'].set_index('location_id').loc[l,'open_date'] < pd.Timestamp('2024-01-01')]

ss_2024 = income_statement(enriched, E, '2024-01-01', '2024-12-31', locations=same_store)
ss_2025 = income_statement(enriched, E, '2025-01-01', '2025-12-31', locations=same_store)
print('same-store 2024 revenue:', f"{ss_2024['revenue']:,.0f}")
print('same-store 2025 revenue:', f"{ss_2025['revenue']:,.0f}")
print('same-store growth:', f"{100*(ss_2025['revenue']/ss_2024['revenue']-1):.1f}%")

same-store 2024 revenue: 105,956,081
same-store 2025 revenue: 115,341,587
same-store growth: 8.9%


In [43]:
for cat in enriched.category.unique():
    sub = enriched[(enriched.channel=='Delivery') & (enriched.category==cat)]
    rev = sub.net_sales.sum(); gp = rev - sub.cogs.sum(); contrib = gp - sub.commission.sum()
    print(f"{cat:12s} delivery contribution margin: {100*contrib/rev:.1f}%")

Coffee       delivery contribution margin: 45.2%
Beverages    delivery contribution margin: 38.7%
Food         delivery contribution margin: 26.2%


In [53]:
"""
CafeCorp -> dashboard data export
Exports at the FINEST grain the dashboard needs, so all aggregation
happens live in JavaScript based on whatever filters are active.
Nothing is pre-aggregated except where the source data is already coarse.
"""
import pandas as pd, json

RAW = 'CafeCorp - Sample Database.xlsx'
SHEETS = ['Locations','Products','Channels','Sales','Tickets','Expenses','Inventory','Targets']
d = {sh: pd.read_excel(RAW, sheet_name=sh) for sh in SHEETS}

# ---------- CLEAN (same decisions as our Python analysis) ----------
P = d['Products'].copy()
P['cost_ratio'] = P.standard_unit_cost / P.unit_price
cat_ratio = P.groupby('category').cost_ratio.median()
P['cost_imputed'] = P.standard_unit_cost.isna()
P['standard_unit_cost'] = P.standard_unit_cost.fillna((P.unit_price * P.category.map(cat_ratio)).round(2))

E = d['Expenses'].copy()
COST_COLS = ['labor_cost','rent','utilities','marketing','other_opex']
for c in COST_COLS:
    E[c + '_imputed'] = E[c].isna()
E['rent'] = E.groupby('location_id')['rent'].transform(lambda x: x.fillna(x.median()))

S = d['Sales'].copy()
S['month'] = S.date.dt.strftime('%Y-%m')
sm = S.groupby(['month','location_id'], as_index=False).net_sales.sum()
E = E.merge(sm, on=['month','location_id'], how='left')
ratio = (E.loc[~E.labor_cost_imputed].assign(r=lambda x: x.labor_cost / x.net_sales)
           .groupby('location_id').r.median())
m = E.labor_cost.isna()
E.loc[m, 'labor_cost'] = (E.loc[m,'net_sales'] * E.loc[m,'location_id'].map(ratio)).round(2)

# ---------- INDEX MAPS (compact encoding: store ints, not repeated strings) ----------
L = d['Locations'].copy()
L['cohort'] = pd.cut(L.open_date,
    bins=[pd.Timestamp('1900-01-01'), pd.Timestamp('2023-12-31'),
          pd.Timestamp('2024-12-31'), pd.Timestamp('2099-01-01')],
    labels=['Same-store','2024 openings','2025 openings'])

months  = sorted(S.month.unique())
locIds  = list(L.location_id)
chanIds = list(d['Channels'].channel)
catIds  = sorted(P.category.unique())
prodIds = list(P.product_id)

mi = {v:i for i,v in enumerate(months)}
li = {v:i for i,v in enumerate(locIds)}
ci = {v:i for i,v in enumerate(chanIds)}
ki = {v:i for i,v in enumerate(catIds)}
pi = {v:i for i,v in enumerate(prodIds)}

# ---------- FACT: sales at month x loc x channel x product ----------
f = S.groupby(['month','location_id','channel','product_id'], as_index=False).agg(
        qty=('quantity','sum'), net=('net_sales','sum'), disc=('discount','sum'))
sales = [[mi[r.month], li[r.location_id], ci[r.channel], pi[r.product_id],
          int(r.qty), round(r.net,2), round(r.disc,2)] for r in f.itertuples()]

# ---------- FACT: tickets at month x loc x channel ----------
T = d['Tickets'].copy()
T['month'] = T.date.dt.strftime('%Y-%m')
tf = T.groupby(['month','location_id','channel'], as_index=False).tickets.sum()
tickets = [[mi[r.month], li[r.location_id], ci[r.channel], int(r.tickets)] for r in tf.itertuples()]

# ---------- FACT: expenses at month x loc (no channel/product dimension!) ----------
exp = [[mi[r.month], li[r.location_id], round(r.labor_cost,2), round(r.rent,2),
        round(r.utilities,2), round(r.marketing,2), round(r.other_opex,2)]
       for r in E.itertuples()]

# ---------- FACT: targets at month x loc x channel x category ----------
G = d['Targets']
tgt = [[mi[r.month], li[r.location_id], ci[r.channel], ki[r.category],
        round(r.sales_target,2), round(r.gp_target,2)] for r in G.itertuples()]

# ---------- FACT: inventory at month x loc x item ----------
I = d['Inventory'].copy()
itemIds = sorted(I.item_id.unique())
ii = {v:i for i,v in enumerate(itemIds)}
itemMeta = I.groupby('item_id').agg(name=('item_name','first'), unit=('unit','first'),
                                     cost=('unit_cost','first')).reset_index()
inv = [[mi[r.month], li[r.location_id], ii[r.item_id], int(r.opening_qty), int(r.purchases_qty),
        int(r.usage_qty), int(r.waste_qty), int(r.closing_qty)] for r in I.itertuples()]

payload = {
  'months': months,
  'locations': [{'id':r.location_id,'name':r.location_name,'city':r.city,'format':r.format,
                 'open':r.open_date.strftime('%Y-%m-%d'),'sqm':int(r.size_sqm),
                 'cohort':str(r.cohort)} for r in L.itertuples()],
  'channels':  [{'name':r.channel,'comm':float(r.commission_rate)} for r in d['Channels'].itertuples()],
  'categories': catIds,
  'products':  [{'id':r.product_id,'name':r.product_name,'cat':ki[r.category],
                 'price':float(r.unit_price),'cost':float(r.standard_unit_cost),
                 'imputed':bool(r.cost_imputed)} for r in P.itertuples()],
  'items':     [{'id':r.item_id,'name':r.name,'unit':r.unit,'cost':float(r.cost)}
                for r in itemMeta.itertuples()],
  'sales': sales, 'tickets': tickets, 'expenses': exp, 'targets': tgt, 'inventory': inv,
}

with open('data_full.json','w') as fh:
    json.dump(payload, fh, separators=(',',':'))

import os
print("sales rows:", len(sales), "| tickets:", len(tickets), "| expenses:", len(exp),
      "| targets:", len(tgt), "| inventory:", len(inv))
print("file size:", round(os.path.getsize('data_full.json')/1024/1024, 2), "MB")

sales rows: 23418 | tickets: 876 | expenses: 292 | targets: 2628 | inventory: 4380
file size: 0.84 MB


In [49]:
kpi_2025 = income_statement(enriched, E, '2025-01-01', '2025-12-31')
kpi_2024 = income_statement(enriched, E, '2024-01-01', '2024-12-31')

import json
print(json.dumps({'current': kpi_2025, 'prior': kpi_2024}))

{"current": {"revenue": 183001462.98, "cogs": 61434534.970000006, "gross_profit": 121566928.00999999, "gross_margin_pct": 66.4294842403997, "commission": 15057291.1694, "contribution": 106509636.84059998, "contribution_pct": 58.20152205681563, "opex": 90171273.64000002, "operating_profit": 16338363.200599968, "operating_margin_pct": 8.927995948527236}, "prior": {"revenue": 124571934.72999999, "cogs": 41657145.830000006, "gross_profit": 82914788.89999998, "gross_margin_pct": 66.55976651539639, "commission": 9203092.555200001, "contribution": 73711696.34479998, "contribution_pct": 59.1719928766976, "opex": 63076321.56, "operating_profit": 10635374.784799978, "operating_margin_pct": 8.537536811843959}}
